# TrustNet existing ViT: baseline and conditional fine-tuning

Attach the private source ZIP and labeled face data. Use a GPU and fresh session. The xhlulu GAN-face dataset is an optional historical benchmark, not modern-generator validation. This notebook does not publish a model or upload private portraits. Fine-tuning is disabled until explicitly selected.


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile, subprocess, importlib.metadata
WORK = Path('/kaggle/working/trustnet-image')
WORK.mkdir(parents=True, exist_ok=True)
# Keep Kaggle's GPU torch/torchvision installation. Install other dependencies before importing them.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=2,<3', 'scipy>=1.14,<2', 'scikit-learn>=1.6,<2', 'pillow>=10', 'PyYAML>=6,<7',
    'python-dotenv>=1,<2', 'pydantic>=2,<3', 'httpx>=0.27,<1',
    'transformers>=4.48,<6', 'huggingface-hub>=0.28,<2'])
opencv_variants = ('opencv-python', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless')
installed = {}
for name in opencv_variants:
    try: installed[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: pass
if installed != {'opencv-python': '4.14.0.94'}:
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', *opencv_variants])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python==4.14.0.94'])
import torch, torchvision, cv2
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator; this notebook must not silently train on CPU.'
assert hasattr(cv2, 'CascadeClassifier') and hasattr(cv2, 'data'), 'Restart this session after installing dependencies.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__, '| OpenCV:', cv2.__version__)


In [ ]:
packages = list(Path('/kaggle/input').rglob('TrustNet-Image-Training-Sources.zip'))
assert len(packages) == 1, 'Attach exactly one source archive dataset.'
SOURCE = WORK / 'source'
assert not SOURCE.exists(), 'Use a fresh working folder; do not overwrite a previous run.'
with zipfile.ZipFile(packages[0]) as archive:
    assert sum(item.file_size for item in archive.infolist()) < 10_000_000
    for item in archive.infolist():
        target = (SOURCE / item.filename).resolve()
        assert target.is_relative_to(SOURCE.resolve()), 'Unsafe archive path'
    archive.extractall(SOURCE)
source_manifest = json.loads((SOURCE / 'training_source_manifest.json').read_text())
for name, expected in source_manifest.items():
    assert hashlib.sha256((SOURCE / name).read_bytes()).hexdigest() == expected, name
os.chdir(SOURCE)
sys.path.insert(0, str(SOURCE))
print('Verified source files:', len(source_manifest))


## Prepare the historical benchmark (optional)
The publisher splits contain older face data with unknown upstream ViT overlap. This is a diagnostic benchmark only. End-to-end manifests retain localization failures; classifier manifests contain usable crops. Use independent recent data before selecting or promoting a policy.


In [ ]:
roots = [p for p in Path('/kaggle/input').rglob('140k-real-and-fake-faces') if p.is_dir()]
assert len(roots) == 1, 'Attach xhlulu/140k-real-and-fake-faces version 2, or replace this cell with your reviewed prepared manifests.'
from models.image_deepfake.training.prepare_kaggle import prepare
PREPARED = WORK / 'prepared-vit'
provenance = prepare(roots[0], PREPARED)
print(provenance['counts'])
print(provenance['limitations'])


## Frozen ViT baseline
This deterministic balanced sample is a smoke evaluation. Increase to the complete independent holdout for a release benchmark. Do not use test results to fit thresholds. Direct full-frame scores are diagnostic; policy scores refer to detected faces. Both reports preserve failures in their denominators.


In [ ]:
import random
os.environ['IMAGE_ALLOW_MODEL_DOWNLOAD'] = 'true'
os.environ['IMAGE_ALLOW_HOSTED_INFERENCE'] = 'false'
rows = json.loads((PREPARED / 'policy_test.json').read_text())
rng = random.Random(42)
sample = []
for label in (0, 1):
    choices = [r for r in rows if r[1] == label]
    sample.extend(rng.sample(choices, min(100, len(choices))))
SAMPLE = WORK / 'vit-smoke-test.json'; SAMPLE.write_text(json.dumps(sample))
from models.image_deepfake.evaluation.benchmark import evaluate
for mode in ('vit_direct', 'vit_policy'):
    report = evaluate(SAMPLE, WORK / (mode+'-baseline.json'), mode=mode, purpose='test',
                      variants=('original','jpeg_75','resize_256'), bootstrap=200)
    print(mode, report['summary'])
print('No validated operating point is installed; policy decisions abstain while raw face scores remain measurable.')


## Conditional training
Use reviewed face crops with train/validation/test source separation and licensed recent data. The training runner checks exact and candidate near-duplicate leakage. A source group does not prove unknown subject identity independence. Historical GAN-only training is not evidence of modern accuracy. Compare head-only and full tuning in separate fresh output directories; do not tune against the final test.


In [ ]:
RUN_TRAINING = False  # Set True only after baseline and reviewed data justify this experiment.
ARTIFACT = WORK / 'vit-candidate'
if RUN_TRAINING:
    from models.image_deepfake.training.train_vit import run
    manifest = run(PREPARED/'train.json', PREPARED/'validation.json', PREPARED/'test.json',
                   ARTIFACT, epochs=10, batch_size=16, seed=42, head_only=True, face_crops_prepared=True)
    print(manifest)
else:
    print('Training skipped. No candidate checkpoint or accuracy improvement is claimed.')


## Independent policy validation and final test
With recent independent all-input manifests, evaluate a candidate using `evaluation.benchmark`, select boundaries using `evaluation.select_policy`, and run the frozen candidate on the final test. `evaluation.promote_policy` rejects overlap and unmet budgets. Never substitute the 200-image smoke sample for these gates. The policy requires at least 1000 independent scored examples per class for selection; final test requires at least 1000 per class. Inspect confidence intervals and critical slices even when automated gates pass.


In [ ]:
if RUN_TRAINING:
    allowed = ['vit_manifest.json','config.json','preprocessor_config.json','model.safetensors',
               'dataset_index.json','training_report.json']
    result_zip = Path('/kaggle/working/TrustNet-ViT-Candidate.zip')
    with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as package:
        for name in allowed: package.write(ARTIFACT/name, name)
    print('Download candidate:', result_zip)
    print('SHA-256:', hashlib.sha256(result_zip.read_bytes()).hexdigest())
    print('Keep training_state.pt separately for optimizer recovery. Installation does not activate a candidate.')


## Local handoff
Install into a new version directory with `scripts/setup/install-image-vit.py --package <zip> --target models/image_deepfake/trained/vit-candidate-1`. Evaluate with `IMAGE_VIT_MODEL_DIR` pointing there. Keep the baseline configuration until independent final-test promotion succeeds. See `docs/image-vit-v3.md` in the local project for exact evaluation and rollback commands. No Kaggle execution is implied by generating this notebook.
